In [0]:
# Databricks notebook source

# ============================================
# PARAMETRO DE ENV VINDO DA PIPELINE / JOB
# ============================================
dbutils.widgets.text("env", "")
p_env = dbutils.widgets.get("env")

env_norm = p_env.strip().lower()
if env_norm in ("hmg", "homol", "hml"):
    env_suffix = "hml"
elif env_norm in ("prod", "prd"):
    env_suffix = "prd"
else:
    env_suffix = "dev"

control_catalog = f"platform_{env_suffix}"

spark.sql(f"""
CREATE OR REPLACE PROCEDURE {control_catalog}.utils.proc_schema_check
(
  IN p_config_name STRING,
  IN p_layer       STRING,
  IN p_system      STRING,
  IN p_env         STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
  DECLARE v_cat STRING;
  DECLARE v_sch STRING;
  DECLARE v_tbl STRING;
  DECLARE v_env STRING;
  DECLARE sync_table_full STRING;
  DECLARE add_stmt STRING;
  DECLARE added_names STRING;

  CALL {control_catalog}.utils.proc_load_config(
    p_config_name,
    p_layer,
    p_system
  );

  SET v_env = CASE
    WHEN LOWER(TRIM(p_env)) IN ('hmg', 'homol', 'hml') THEN 'hml'
    WHEN LOWER(TRIM(p_env)) IN ('prod', 'prd') THEN 'prd'
    ELSE LOWER(TRIM(p_env))
  END;

  SET v_cat = (
    SELECT COALESCE(NULLIF(TRIM(cfg_target.target_catalog), ''), '{control_catalog}')
    FROM cfg
    LIMIT 1
  );

  SET v_sch = (
    SELECT NULLIF(TRIM(cfg_target.target_schema), '')
    FROM cfg
    LIMIT 1
  );

  SET v_tbl = (
    SELECT NULLIF(TRIM(cfg_target.target_object), '')
    FROM cfg
    LIMIT 1
  );

  SET v_cat = CASE
    WHEN v_cat RLIKE '^(?i).+_(dev|hml|prd)$' THEN v_cat
    ELSE CONCAT(v_cat, '_', v_env)
  END;

  IF v_sch IS NULL OR v_tbl IS NULL THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_schema_check: invalid target';
  END IF;

  SET sync_table_full = CONCAT('`', v_cat, '`.`', v_sch, '`.`', v_tbl, '`');

  CREATE OR REPLACE TEMP VIEW desired_schema AS
  SELECT
    LOWER(e.key) AS col_name_lc,
    e.key AS col_name_raw,
    UPPER(TRIM(e.value)) AS data_type_raw
  FROM cfg
  LATERAL VIEW explode(map_entries(cfg_target.target_schema_definition)) t AS e;

  EXECUTE IMMEDIATE 'USE CATALOG `' || v_cat || '`';

  CREATE OR REPLACE TEMP VIEW current_schema AS
  SELECT
    LOWER(column_name) AS col_name_lc,
    column_name AS col_name_raw,
    UPPER(data_type) AS data_type_raw
  FROM information_schema.columns
  WHERE table_schema = v_sch
    AND table_name = v_tbl;

  CREATE OR REPLACE TEMP VIEW to_add AS
  SELECT d.col_name_raw, d.data_type_raw
  FROM desired_schema d
  LEFT JOIN current_schema c
    ON d.col_name_lc = c.col_name_lc
  WHERE c.col_name_lc IS NULL;

  CREATE OR REPLACE TEMP VIEW v_add_parts AS
  SELECT
    CASE
      WHEN COUNT(*) = 0 THEN NULL
      ELSE CONCAT(
        'ALTER TABLE ', sync_table_full, ' ADD COLUMNS (',
        ARRAY_JOIN(
          SORT_ARRAY(COLLECT_LIST(CONCAT('`', col_name_raw, '` ', data_type_raw))),
          ', '
        ),
        ')'
      )
    END AS add_stmt,
    CASE
      WHEN COUNT(*) = 0 THEN ''
      ELSE ARRAY_JOIN(SORT_ARRAY(COLLECT_LIST(col_name_raw)), ', ')
    END AS names
  FROM to_add;

  SET add_stmt = COALESCE((SELECT add_stmt FROM v_add_parts), 'SELECT 1');
  SET added_names = COALESCE((SELECT names FROM v_add_parts), '');

  EXECUTE IMMEDIATE add_stmt;

  SELECT CASE
    WHEN added_names <> '' THEN CONCAT('Added columns: ', added_names)
    ELSE 'No columns added'
  END AS message;
END;
""")